# HumanEval GRPO - Qwen3-8B

**Tinker RL Project — PES University MTech Capstone (Group 6)**

| Field | Value |
|-------|-------|
| **Model** | `Qwen/Qwen3-8B` |
| **Benchmark** | HumanEval (164 coding problems) |
| **Method** | GRPO + LoRA rank 32 |
| **Reward** | Binary: 1.0 if generated code passes all unit tests, 0.0 otherwise |
| **Baseline** | Madhu's result: 32% → 40% pass@1 on Qwen3-3.8B (50 steps) |
| **Target** | 40% → 50%+ pass@1 on Qwen3-8B |
| **Training API** | Tinker (cloud GPU) |
| **Status** | Ready to run |

In [ ]:
!pip install -q atroposlib==0.3.0 \
    'git+https://github.com/thinking-machines-lab/tinker.git' \
    datasets transformers pydantic python-dotenv

In [ ]:
!git clone https://github.com/pes-llm-research/tinker-rl-lab.git
%cd tinker-rl-lab/atropos
!pip install -e . -q

In [ ]:
import os
os.environ['TINKER_API_KEY'] = 'YOUR_TINKER_API_KEY'
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN'
os.environ['TINKER_CONFIG_PATH'] = 'configs/humaneval_qwen_8b.yaml'
print('Credentials set.')

In [ ]:
!nohup python -m atroposlib.server > /tmp/atropos_server.log 2>&1 &
import time; time.sleep(5)

!nohup python -m tinker_atropos.environments.humaneval_tinker \
    --config configs/humaneval_qwen_8b.yaml \
    > /tmp/humaneval_env.log 2>&1 &
time.sleep(15)
!tail -5 /tmp/humaneval_env.log

In [ ]:
# Launch trainer
!python launch_training.py --config configs/humaneval_qwen_8b.yaml

In [ ]:
# RESULTS — paste step data here after training completes
steps = []   # TODO
rewards = [] # TODO  (pass@1 rate per step)

# Baseline comparison
baseline = {'model': 'Qwen3-3.8B (Madhu, 50 steps)', 'initial': 0.32, 'final': 0.40}

if steps and rewards:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(steps, rewards, color='#e67e22', linewidth=2, label='Qwen3-8B (this run)')
    ax.axhline(y=baseline['initial'], color='gray', linestyle='--', alpha=0.5, label=f"Baseline start: {baseline['initial']:.0%}")
    ax.axhline(y=baseline['final'], color='gray', linestyle='-', alpha=0.5, label=f"Baseline end: {baseline['final']:.0%}")
    ax.set_xlabel('Training Step')
    ax.set_ylabel('pass@1')
    ax.set_title('HumanEval GRPO - Qwen3-8B\npass@1 Trajectory')
    ax.set_ylim(-0.05, 1.05)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f'Initial: {rewards[0]:.2%} | Final: {rewards[-1]:.2%}')
    print(f'Improvement vs baseline: +{rewards[-1] - baseline["final"]:.2%}')
else:
    print('Run training first.')